In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import configparser
from PIL import Image
from pathlib import Path
import os

class EyeDataset(torch.utils.data.dataset.Dataset):
  
  def __init__(self, partition, mean, std):
    '''
    - partition: 'training' o 'validation'
    - mean: media para normalizar los datos
    - std: desvío estándar
    '''
    super(EyeDataset, self).__init__()

    # chequeo de que no entre cualquier cosa en vez de partition
    assert partition in ['training', 'validation'], 'partition has to be training or validation, not {}'.format(partition)

    # transformaciones
    self.data_transforms = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize(mean=mean, std=std)])
    
    self.data_path = Path(os.getcwd()).parent

    # listado de imagenes para cada dataset
    data_config = configparser.ConfigParser()
    data_config.read(self.data_path/'data/splits/global_binaria.ini')

    # listado de labels para cada imagen
    labels_config = configparser.ConfigParser()
    labels_config.read(self.data_path/'data/labels/global_labels.ini')

    self.data = [(image,
                  int(labels_config['label'].get(image)))
                  for image in data_config['split'].get(partition).split(sep=',')
                  ]
    
    # hacemos una lista de nombres para las clases
    self.class_names = ['buena', 'mala']

    # Mapeo de las clases
    self.label_map = {'buena': 0, 'mala': 1}

  def __len__(self):
    '''
    Método que devuelve la cantidad de elementos del dataset
    '''
    return len(self.data)
  
  def __getitem__(self, index):
    '''
    Método que devuelve el elemento index del dataset
    '''
    # obtengo la ruta de la imagen y su etiqueta asociada
    img_path, label = self.data[index]

    # cargo la imagen
    img = Image.open(self.data_path/'images/'/img_path)
    # transformaciones
    img = self.data_transforms(img)

    # convierto la etiqueta en un tensor de tipo long
    label = torch.tensor(label, type=torch.long())

    # retornamos el par
    return (img, label)